# 🎧 Higgs TTS 3 — Colab benchmark (CUDA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/higgs_colab_benchmark.ipynb)

Стенд для русского **TTS** на серверном GPU, со сравнением против локального замера на
Apple Silicon M1. STT идёт вторым этапом. Блокнот следует правилам `AGENTS.md`: **ни один
результат не печатается, если операция не выполнялась** — незапущенный или недоступный
этап отмечается `SKIPPED`/`FAILED` с причиной.

## Автоотключение и сохранность данных

ВМ **отключается по завершении безусловно** — квота Colab не должна гореть на простое.
Поэтому всё, что нужно для разбора результатов, пишется на Google Drive **по ходу**, а не
в конце: метрики каждого этапа, логи установки, лог сервера, транскрипт, WAV-файлы и
полный вывод всех дочерних процессов (`metrics/session.log`).

По той же причине этапы **не бросают исключений**: любой отказ записывается как статус с
трассировкой, иначе `Run all` остановился бы на ошибке и до отключения ВМ дело не дошло
бы. Смотреть результаты нужно на Диске, а не в выводе ячеек.

## Изоляция памяти

Каждая модель работает **отдельным процессом**. Ядро блокнота никогда не держит ссылку
на модель, поэтому VRAM и хостовая RAM освобождаются завершением процесса, а не вызовом
`del`. Сервер TTS запускается своей группой процессов и гасится вместе с ней.

## Два бэкенда TTS

У чекпоинта `bosonai/higgs-tts-3-4b` нет собственного кода, а `higgs_multimodal_qwen3`
не реализован в `transformers` — пути «`from_pretrained` + `generate`» не существует.
Модель реализуют два стека, и раннер умеет оба:

| Бэкенд | Команда | Состояние |
|---|---|---|
| `vllm` (по умолчанию) | `vllm serve --omni` из [vllm-omni](https://github.com/vllm-project/vllm-omni) | Python 3.13, этапы Higgs идут eager, ядра flashinfer CuTe не задействованы |
| `sglang` | `sgl-omni serve` — путь из model card | На Tesla T4 загружает веса и падает на захвате CUDA-графа: `KeyError: 'sm_75'` |

Ни один из стеков не заявляет поддержку Turing, поэтому T4 остаётся экспериментом. Но в
отличие от SGLang, у vLLM-пути заранее не видно того, что его исключает: именно та фаза,
которая убила `sgl-omni`, здесь не выполняется.

Локальный замер на M1 идёт через MLX-Audio — независимую реализацию на MLX. Работающий
синтез на Metal ничего не говорит о CUDA: это разный код.

## 1. Настройки запуска

In [ ]:
# ── Что запускать ───────────────────────────────────────────
RUN_TTS = True             # основная цель проекта
RUN_STT = True             # вторичный этап
RUN_QWEN = True            # диагностический контроль T4 + альтернатива для аудиокниг (#52)

FORCE_HIGGS_ON_T4 = False  # Higgs на Tesla T4 репродуцируемо не работает (#48), поэтому
                           # на T4 его этап по умолчанию пропускается. True — запустить
                           # всё равно, зная, что это заведомо ожидаемый провал по #48.

USE_DRIVE = True           # хранить входы/выходы на Google Drive

REPO_URL = "https://github.com/vedmalex/higgs-local-test.git"
REPO_REF = "main"          # ветка/тег/SHA репозитория с раннерами

# ── TTS ─────────────────────────────────────────────────────
TTS_BACKEND = "vllm"       # "vllm" | "sglang"
INSTALL_TTS_STACK = True   # ставить стек (долго: гигабайты колёс)
TTS_MAX_NEW_TOKENS = 2048
TTS_MEM_FRACTION_STATIC = None   # доля VRAM под статический пул, напр. 0.85
TTS_MIN_CAPABILITY = None        # напр. "8.9" — пропустить TTS на более старом GPU
TTS_SERVER_ARGS = []             # доп. аргументы сервера, напр. ["max-model-len=4096"];
                                 # для sglang при падении на графе: ["talker-cuda-graph=off"]
TTS_SERVER_ENV = []              # доп. переменные окружения сервера
TTS_DEPLOY_CONFIG = None         # свой deploy YAML для vllm; ниже compute 8.0 раннер
                                 # сам берёт configs/higgs_multimodal_qwen3_turing.yaml
TTS_PYTHON = None                # только для sglang: он требует Python <3.13, поэтому на
                                 # Colab нужен отдельный интерпретатор — укажите "3.12"

# ── Qwen3-TTS (#52) ─────────────────────────────────────────
# Второй, независимый TTS-бэкенд — не замена Higgs. Один запуск обслуживает
# один вариант чекпоинта (Base/CustomVoice/VoiceDesign); список ниже запускает
# каждый вариант по очереди отдельным сервером, потому что `task_type`
# работает только с тем чекпоинтом, для которого он обучен. Та же
# анти-фальшпозитив проверка WAV, что у Higgs (src/tts_cuda_common.py).
QWEN_MODEL_VARIANTS = ["0.6b-base", "0.6b-customvoice"]  # Phase 1 (T4-диагностика,
                                 # модели из issue #52); добавьте "1.7b-base",
                                 # "1.7b-customvoice", "1.7b-voicedesign" для Phase 2
                                 # (аудиокнижные возможности)
INSTALL_QWEN_STACK = True
QWEN_MAX_NEW_TOKENS = 2048
QWEN_VOICE = "vivian"            # предустановленный тембр CustomVoice
QWEN_MEM_FRACTION_STATIC = None
QWEN_MIN_CAPABILITY = None       # напр. "8.9" — пропустить Qwen на более старом GPU
QWEN_SERVER_ARGS = []
QWEN_SERVER_ENV = []
QWEN_DEPLOY_CONFIG = None        # ниже compute 8.0 раннер сам берёт
                                 # configs/qwen3_tts_turing.yaml, если он есть

# ── STT ─────────────────────────────────────────────────────
STT_DTYPE = "float16"      # float16 | bfloat16 | float32

### Токен Hugging Face (не обязателен)

Без токена Хаб отвечает на анонимные запросы: скачивание весов заметно медленнее и
попадает под общие рейт-лимиты (`You are sending unauthenticated requests...` в логе
установки). Токен на чтение убирает обе проблемы и больше ни на что не влияет —
метрики бенчмарка от него не зависят, и при отказе от ввода блокнот работает как раньше.

Ячейка ниже берёт токен из **Colab Secrets** (иконка ключа на левой панели Colab: имя
`HF_TOKEN`, тумблер *Notebook access*), затем из переменной окружения `HF_TOKEN`
(**Kaggle**: Add-ons → Secrets, там же включается доступ ноутбука к секрету), и только
потом спрашивает его вручную. Ввод скрытый, сам токен никуда не печатается и на Диск
не пишется. Пустой ввод (просто Enter) — продолжить без токена.

In [ ]:
import getpass, os


def resolve_hf_token():
    """Токен HF: Colab Secrets → переменная окружения → скрытый ручной ввод.

    Токен не обязателен: без него скачивание идёт неаутентифицированно. Значение
    нигде не печатается, чтобы не утекло в вывод ячейки, который сохраняется в .ipynb.
    """
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token and token.strip():
            return token.strip(), "Colab Secrets"
    except Exception:
        # Не Colab, секрет не задан или доступ ноутбуку к нему не разрешён —
        # это не ошибка: ниже есть ещё два источника.
        pass

    token = os.environ.get("HF_TOKEN")
    if token and token.strip():
        return token.strip(), "переменная окружения HF_TOKEN"

    try:
        token = getpass.getpass("HF_TOKEN (Enter, чтобы продолжить без токена): ")
    except Exception as error:
        # Неинтерактивный запуск (papermill, `Run all` без stdin) не должен ронять прогон.
        print(f"ℹ️  интерактивный ввод недоступен: {error!r}")
        token = ""
    if token and token.strip():
        return token.strip(), "интерактивный ввод"
    return None, None


HF_TOKEN, HF_TOKEN_SOURCE = resolve_hf_token()
if HF_TOKEN:
    # huggingface_hub читает HF_TOKEN из окружения сам; дочерние процессы (раннеры
    # TTS/STT, серверы vLLM) наследуют окружение ядра, поэтому этого достаточно.
    os.environ["HF_TOKEN"] = HF_TOKEN
    print(f"✅ HF_TOKEN установлен ({HF_TOKEN_SOURCE}) — скачивание аутентифицированное.")
else:
    print("ℹ️  HF_TOKEN не задан: скачивание пойдёт неаутентифицированно — медленнее и "
          "под общими рейт-лимитами Хаба.\n"
          "    На сами метрики это не влияет, этапы выполняются как обычно.")

In [ ]:
import json, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

SESSION_LOG = None   # выставляется после монтирования Drive

def _open_logs(paths):
    handles = []
    for path in paths:
        if path is None:
            continue
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        handles.append(Path(path).open("a", encoding="utf-8"))
    return handles

def run(command, log=None, **kwargs):
    """Запуск дочернего процесса с потоковым выводом. Возвращает returncode.

    Вывод дублируется в `metalog`-файлы на Диске: вывод ячейки исчезнет вместе с
    ВМ, которая отключается безусловно, поэтому единственная долговечная копия —
    та, что лежит на Google Drive.
    """
    printable = command if isinstance(command, str) else " ".join(shlex.quote(c) for c in command)
    print(f"$ {printable}")
    handles = _open_logs([SESSION_LOG, log])
    for handle in handles:
        handle.write(f"$ {printable}\n")
    try:
        process = subprocess.Popen(command, shell=isinstance(command, str),
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, **kwargs)
        for line in process.stdout:
            print(line, end="")
            for handle in handles:
                handle.write(line)
        returncode = process.wait()
        for handle in handles:
            handle.write(f"[exit {returncode}]\n")
        return returncode
    finally:
        for handle in handles:
            handle.close()

def note(message: str) -> None:
    """Печать с копией в session.log на Диске."""
    print(message)
    for handle in _open_logs([SESSION_LOG]):
        handle.write(message + "\n")
        handle.close()

def python_paths(interpreter) -> list:
    """Пути импорта, которые реально видит указанный интерпретатор."""
    probe = "import json, sys; print(json.dumps(sys.path))"
    result = subprocess.run([str(interpreter), "-c", probe], capture_output=True, text=True)
    result.check_returncode()
    return json.loads(result.stdout)

def link_system_packages(venv: Path) -> None:
    """Открыть venv доступ к системным пакетам Colab.

    Colab держит пакеты в `dist-packages`, и одного `--system-site-packages` мало:
    Debian-раскладка отличается от той, которую ожидает `site.py` внутри venv.
    Поэтому пути родительского интерпретатора добавляются через `.pth`-файл. Он
    дописывается в КОНЕЦ `sys.path`, поэтому пакеты самого venv сохраняют
    приоритет над системными — на этом держится разделение окружений.
    """
    interpreter = venv / "bin/python"
    inherited = [p for p in python_paths(sys.executable)
                 if p and Path(p).is_dir() and not str(p).startswith(str(venv))]
    site_dir = subprocess.run(
        [str(interpreter), "-c", "import site; print(site.getsitepackages()[0])"],
        capture_output=True, text=True, check=True).stdout.strip()
    pth = Path(site_dir) / "_colab_system_packages.pth"
    pth.write_text("\n".join(inherited) + "\n", encoding="utf-8")
    print(f"Системные пути Colab подключены через {pth} ({len(inherited)} каталогов)")

def ensure_venv(venv: Path, steps: list, python_version=None, installer="pip",
                log=None) -> None:
    """Создать окружение и выполнить в нём шаги установки, если это ещё не сделано.

    `steps` — список групп аргументов для `pip install`/`uv pip install`, по одной
    группе на команду: порядок бывает важен (vLLM ставится до vllm-omni).

    Два обхода особенностей Colab:

    * `python -m venv` падает на `ensurepip`: Debian выносит его из стандартной
      поставки. Поэтому venv создаётся с `--without-pip`, а pip берётся
      системный — запущенный интерпретатором venv, он ставит пакеты именно в
      этот venv, потому что `sys.prefix` указывает на него.
    * `--system-site-packages` вместе с `link_system_packages()` даёт доступ к
      предустановленным torch и pip, ничего не переустанавливая.

    `installer="uv"` нужен там, где так предписывает документация пакета: uv умеет
    `--torch-backend=auto` и приносит собственный интерпретатор, если версия Colab
    пакету не подходит. Такое окружение полностью самостоятельное — свой torch.

    Вывод установки НЕ подавляется: `-q` однажды уже скрыл настоящую причину
    отказа, оставив в логе только «See above for output».

    Готовность отмечается файлом-маркером: неполное окружение от прошлого
    запуска пересоздаётся, а не используется молча.
    """
    marker = venv / ".higgs-ready"
    if marker.exists():
        print(f"Окружение готово: {venv}")
        return
    if venv.exists():
        print(f"Окружение {venv} неполное — пересоздаём")
        shutil.rmtree(venv)
    if log is not None:
        Path(log).unlink(missing_ok=True)   # лог относится к одной попытке

    if installer == "uv":
        if run([sys.executable, "-m", "pip", "install", "-q", "uv"], log=log) != 0:
            raise RuntimeError("не удалось установить uv")
        environment = {**os.environ, "UV_PYTHON_DOWNLOADS": "automatic"}
        version = python_version or f"{sys.version_info.major}.{sys.version_info.minor}"
        if run([sys.executable, "-m", "uv", "venv", "--python", version, str(venv)],
               log=log, env=environment) != 0:
            raise RuntimeError(f"uv не смог создать окружение на Python {version}")
        for step in steps:
            if run([sys.executable, "-m", "uv", "pip", "install",
                    "--python", str(venv / "bin/python"), *step],
                   log=log, env=environment) != 0:
                raise RuntimeError(f"установка {step} не удалась — см. лог")
    else:
        if run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip",
                str(venv)]) != 0:
            raise RuntimeError(f"не удалось создать окружение {venv}")
        link_system_packages(venv)
        for step in steps:
            if run([str(venv / "bin/python"), "-m", "pip", "install", *step],
                   log=log) != 0:
                raise RuntimeError(f"установка {step} не удалась — см. лог")
    marker.write_text("ok\n", encoding="utf-8")

# ── Рабочая область ─────────────────────────────────────────
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/higgs-benchmark")
else:
    WORKSPACE = Path("/content/higgs-benchmark")

SAMPLES_DIR = WORKSPACE / "samples"
OUTPUT_DIR = WORKSPACE / "output"
METRICS_DIR = WORKSPACE / "metrics"
for directory in (SAMPLES_DIR, OUTPUT_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# С этого момента весь вывод дочерних процессов пишется и на Диск.
SESSION_LOG = METRICS_DIR / "session.log"
SESSION_LOG.write_text("", encoding="utf-8")

# Веса живут на локальном диске ВМ, а не на Drive: 15 ГБ через FUSE
# загружаются несравнимо медленнее и съедают квоту Диска.
HF_HOME = Path("/content/hf-cache")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)

REPO = Path("/content/higgs-local-test")
if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)])
REPO_SHA = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                          capture_output=True, text=True).stdout.strip()

print(f"\nРабочая область: {WORKSPACE}")
print(f"Кэш весов:       {HF_HOME}")
print(f"Раннеры:         {REPO} @ {REPO_SHA[:12]}")

for directory, label in ((SAMPLES_DIR, "Входные сэмплы"), (OUTPUT_DIR, "Результаты")):
    files = [f for f in sorted(directory.iterdir()) if f.is_file() and not f.name.startswith(".")]
    print(f"\n{label} ({directory}): {len(files)} файлов")
    for f in files:
        print(f"  - {f.name} ({f.stat().st_size / (1024 ** 2):.2f} MB)")

## 2. GPU и входные данные

Отсутствующий вход даёт `SKIPPED`: синтетический сигнал вместо записи речи не
подставляется, потому что метрики по нему ничего не измеряют.

In [ ]:
run(["nvidia-smi"])

import torch  # предустановлен в Colab; переустановка сломала бы связку CUDA/torchvision

GPU = None
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU = {"name": props.name, "capability": (props.major, props.minor),
           "total_memory_gb": props.total_memory / (1024 ** 3),
           "bf16": torch.cuda.is_bf16_supported()}
    print(f"\nGPU: {GPU['name']} | compute {props.major}.{props.minor} | "
          f"VRAM {GPU['total_memory_gb']:.1f} GB | bf16: {GPU['bf16']}")
    print(f"torch: {torch.__version__} | python: {sys.version.split()[0]}")
else:
    print("\n⚠️  CUDA-устройство недоступно: Runtime → Change runtime type → GPU")

# ── Вход для TTS ─────────────────────────────────────────────
TTS_TEXT = SAMPLES_DIR / "tts_ru.txt"
if not TTS_TEXT.exists():
    TTS_TEXT = REPO / "samples/tts_ru.txt"
REF_WAV, REF_TXT = SAMPLES_DIR / "reference.wav", SAMPLES_DIR / "reference.txt"
print(f"\nTTS текст: {TTS_TEXT}")
print("Клонирование голоса: "
      + ("эталон найден" if REF_WAV.exists() and REF_TXT.exists()
         else f"SKIPPED — нужны {REF_WAV.name} и {REF_TXT.name} в {SAMPLES_DIR}"))

# ── Вход для STT ─────────────────────────────────────────────
STT_INPUT = SAMPLES_DIR / "stt_ru.wav"
STT_REFERENCE = SAMPLES_DIR / "stt_ru.txt"   # точная расшифровка для WER
if not STT_INPUT.exists():
    print(f"\nℹ️  STT будет SKIPPED: положите русскую запись в {STT_INPUT}")
else:
    print(f"\nSTT вход: {STT_INPUT}")
    if STT_REFERENCE.exists():
        print(f"WER считается против {STT_REFERENCE}")
    else:
        print("WER не считается: нет точной расшифровки. Сравнение произвольной записи "
              "со встроенным фикстур-текстом дало бы число, которое ничего не измеряет. "
              f"Положите расшифровку в {STT_REFERENCE}.")

In [ ]:
# ── Гейт Higgs по типу GPU (#48) ─────────────────────────────
# Higgs на Tesla T4 репродуцируемо НЕ работает: сервер поднимается и отвечает
# формально валидным WAV, но с константным сигналом -32768 вместо речи. Это
# задокументированная деградация (issue #48), а не гипотеза, поэтому по умолчанию
# этап не запускается: он сжёг бы десятки минут установки стека и квоту ВМ ради
# заведомо известного провала. RUN_TTS остаётся тем, что задал пользователь, —
# фактическое решение живёт в EFFECTIVE_RUN_TTS.
#
# Гейт касается ТОЛЬКО Higgs. Qwen3-TTS (RUN_QWEN) запускается независимо от типа
# GPU: он и нужен как контроль того, специфична ли #48 для Higgs Code2Wav.
# На не-T4 GPU (Ampere/Hopper/L4 и т.п.) поведение не меняется.
IS_T4 = bool(GPU) and ("T4" in GPU["name"] or GPU["capability"] == (7, 5))

EFFECTIVE_RUN_TTS = RUN_TTS
HIGGS_GATE_REASON = None

if IS_T4:
    print(f"\nОбнаружен {GPU['name']} (compute {GPU['capability'][0]}.{GPU['capability'][1]}).")
    if not RUN_TTS:
        print("Higgs и так отключён: RUN_TTS=False.")
    elif FORCE_HIGGS_ON_T4:
        print("⚠️  FORCE_HIGGS_ON_T4=True — Higgs запускается на T4 принудительно.\n"
              "    Это ЗАВЕДОМО ожидаемый провал по issue #48: ответ придёт валидным\n"
              "    WAV с константным сигналом -32768 вместо речи. Такой прогон имеет\n"
              "    смысл только как перепроверка деградации, а не как замер метрик.")
    else:
        HIGGS_GATE_REASON = (
            f"{GPU['name']}: Higgs репродуцируемо не работает на T4 — issue #48 "
            "(валидный WAV с константным сигналом -32768 вместо речи). "
            "Переопределение: FORCE_HIGGS_ON_T4 = True в ячейке настроек")
        EFFECTIVE_RUN_TTS = False
        print(f"⏭️  Этап Higgs пропускается — {HIGGS_GATE_REASON}.\n"
              "    https://github.com/vedmalex/higgs-local-test/issues/48")
    print(f"Qwen3-TTS гейт не затрагивает: RUN_QWEN={RUN_QWEN}.")

print(f"\nHiggs TTS: RUN_TTS={RUN_TTS} → EFFECTIVE_RUN_TTS={EFFECTIVE_RUN_TTS}")

## 3. TTS: `bosonai/higgs-tts-3-4b`

`src/tts_cuda.py` поднимает выбранный сервер отдельной группой процессов, синтезирует
базовую речь, речь с тегами эмоций и клонированный голос, затем гасит сервер — вместе с
ним уходит вся занятая VRAM. Пик по устройству замеряется через `nvidia-smi`, потому что
веса живут в процессе сервера, а не в раннере.

Установка стека занимает десятки минут и несколько гигабайт.

In [ ]:
TTS_VENV = Path(f"/content/venv-tts-{TTS_BACKEND}")
TTS_BIN = TTS_VENV / "bin"
TTS_PY = TTS_BIN / "python"
TTS_EXE = TTS_BIN / ("vllm" if TTS_BACKEND == "vllm" else "sgl-omni")
# Порядок и флаги — из документации vllm-omni: vLLM ставится первым, с
# --torch-backend=auto, потому что колёса 0.26.0 по умолчанию собраны под CUDA 13,
# а vllm-omni самого vllm в зависимостях не объявляет.
TTS_INSTALL_STEPS = {
    "vllm": [["vllm==0.26.0", "--torch-backend=auto"], ["vllm-omni==0.26.0"]],
    "sglang": [["sglang-omni==0.1.3"]],
}[TTS_BACKEND]
TTS_INSTALLER = "uv" if TTS_BACKEND == "vllm" else "pip"
TTS_INSTALL_LOG = METRICS_DIR / f"tts_install_{TTS_BACKEND}.log"

tts_gate = None
tts_python = None
if not EFFECTIVE_RUN_TTS:
    # HIGGS_GATE_REASON заполнен, только если этап снят гейтом T4 (#48);
    # иначе этап отключён самим пользователем.
    tts_gate = HIGGS_GATE_REASON or "RUN_TTS=False"
elif GPU is None:
    tts_gate = "нет CUDA-устройства"
elif TTS_MIN_CAPABILITY and GPU["capability"] < tuple(
        int(part) for part in f"{TTS_MIN_CAPABILITY}.0".split(".")[:2]):
    tts_gate = (f"{GPU['name']} имеет compute {GPU['capability'][0]}.{GPU['capability'][1]}, "
                f"ниже заданного TTS_MIN_CAPABILITY={TTS_MIN_CAPABILITY}")
elif TTS_BACKEND == "sglang" and sys.version_info >= (3, 13):
    # sglang-omni объявляет Requires-Python >=3.10,<3.13, поэтому pip откажет ещё
    # до скачивания колёс. vllm-omni допускает <3.14 и этой проблемы не имеет.
    if TTS_PYTHON:
        tts_python = TTS_PYTHON
        print(f"sglang-omni не поддерживает Python {sys.version_info.major}."
              f"{sys.version_info.minor}; uv принесёт отдельный {tts_python}.")
    else:
        tts_gate = (f"sglang-omni требует Python <3.13, а в Colab "
                    f"{sys.version_info.major}.{sys.version_info.minor}. Укажите "
                    "TTS_PYTHON = '3.12' или используйте TTS_BACKEND = 'vllm'.")

if tts_gate:
    print(f"TTS стек не устанавливается: {tts_gate}")
elif not INSTALL_TTS_STACK and not TTS_EXE.exists():
    tts_gate = f"окружение {TTS_BACKEND} отсутствует, а INSTALL_TTS_STACK=False"
    print(f"TTS: {tts_gate}")
else:
    try:
        ensure_venv(TTS_VENV, TTS_INSTALL_STEPS, python_version=tts_python,
                    installer=TTS_INSTALLER, log=TTS_INSTALL_LOG)
    except Exception as error:
        tts_gate = f"установка стека {TTS_BACKEND} не удалась: {error!r}"
        print(f"\n❌ {tts_gate}")
    else:
        if TTS_EXE.exists():
            print(f"\n✅ Стек {TTS_BACKEND} установлен: {TTS_EXE}")
        else:
            tts_gate = f"установка прошла, но {TTS_EXE.name} не появился в {TTS_BIN}"
            print(f"\n❌ {tts_gate}")

In [ ]:
TTS_METRICS = METRICS_DIR / f"tts_{TTS_BACKEND}.json"
tts_status = "NOT RUN"

if tts_gate:
    tts_status = f"SKIPPED ({tts_gate})"
    skip_report = {"test": "tts_cuda", "backend": TTS_BACKEND, "status": "SKIPPED",
                   "reason": tts_gate, "results": []}
    if TTS_INSTALL_LOG.exists():
        # Хвост лога попадает в отчёт: «установка не удалась» без него —
        # такое же бесполезное утверждение, как отчёт без причины.
        skip_report["install_log"] = str(TTS_INSTALL_LOG)
        skip_report["install_log_tail"] = TTS_INSTALL_LOG.read_text(
            errors="replace").splitlines()[-60:]
    TTS_METRICS.write_text(json.dumps(skip_report, ensure_ascii=False, indent=2),
                           encoding="utf-8")
else:
    command = [str(TTS_PY), "src/tts_cuda.py",
               "--backend", TTS_BACKEND,
               "--output-dir", str(OUTPUT_DIR),
               "--metrics", str(TTS_METRICS),
               "--text-file", str(TTS_TEXT),
               "--max-new-tokens", str(TTS_MAX_NEW_TOKENS)]
    if REF_WAV.exists() and REF_TXT.exists():
        command += ["--ref-audio", str(REF_WAV), "--ref-text", str(REF_TXT)]
    if TTS_MEM_FRACTION_STATIC is not None:
        command += ["--mem-fraction-static", str(TTS_MEM_FRACTION_STATIC)]
    if TTS_DEPLOY_CONFIG:
        command += ["--deploy-config", str(TTS_DEPLOY_CONFIG)]
    for server_arg in TTS_SERVER_ARGS:
        command += ["--server-arg", server_arg]
    for server_env in TTS_SERVER_ENV:
        command += ["--server-env", server_env]

    # PATH нужен, чтобы раннер нашёл исполняемый файл сервера своего venv.
    environment = {**os.environ, "PATH": f"{TTS_BIN}:{os.environ['PATH']}"}
    returncode = run(command, cwd=str(REPO), env=environment)
    tts_status = "PASSED" if returncode == 0 else f"FAILED (exit {returncode})"

print(f"\nTTS: {tts_status}")
run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

In [ ]:
from IPython.display import Audio, display

def read_metrics(path):
    """Метрики читаются мягко: неудача чтения не должна прерывать Run all."""
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception as error:
        note(f"не удалось прочитать {path}: {error!r}")
        return None

tts_metrics = read_metrics(TTS_METRICS) if TTS_METRICS.exists() else None

if not tts_metrics:
    print(f"TTS метрик нет: {tts_status}")
elif tts_metrics.get("status") == "SKIPPED":
    print(f"TTS SKIPPED: {tts_metrics['reason']}")
    for line in tts_metrics.get("install_log_tail", [])[-20:]:
        print("  " + line)
else:
    if tts_metrics.get("status") == "FAILED":
        print(f"TTS FAILED: {tts_metrics.get('exception')}")
        server_log = Path(tts_metrics.get("server_log")
                          or (OUTPUT_DIR / f"{TTS_BACKEND}_server.log"))
        if server_log.exists():
            print(f"\n--- последние строки {server_log.name} ---")
            print("\n".join(server_log.read_text(errors="replace").splitlines()[-40:]))

    for result in tts_metrics.get("results", []):
        if result["status"] != "PASSED":
            print(f"{result['name']}: {result['status']} — "
                  f"{result.get('reason') or result.get('exception')}")
            continue
        print(f"{result['name']}: {result['audio_duration_seconds']:.2f} с аудио за "
              f"{result['processing_seconds']:.2f} с → RTF {result['rtf']:.3f}")
        display(Audio(filename=result["output"]))

## 3b. TTS-контроль: Qwen3-TTS (#52)

Второй, независимый TTS-бэкенд на том же T4 — диагностический контроль (проверяет,
воспроизводится ли деградация #48 в целом на связке T4/vLLM-Omni, или она специфична
для Higgs Code2Wav) и отдельный кандидат для аудиокниг. `src/tts_qwen_cuda.py`
поднимает `vllm serve --omni` для одного варианта чекпоинта Qwen3-TTS
(Base/CustomVoice/VoiceDesign) за раз, синтезирует то, что поддерживает именно этот
вариант, и гасит сервер. **Higgs не заменяется**: результаты хранятся раздельно
(`metrics/tts_qwen_<variant>.json`), провал одного бэкенда никогда не выдаётся за
проходной результат другого. Источники и обоснование каждого технического решения —
[`docs/research/qwen3-tts-notes.md`](https://github.com/vedmalex/higgs-local-test/blob/main/docs/research/qwen3-tts-notes.md).


In [ ]:
QWEN_VENV = Path("/content/venv-tts-qwen")
QWEN_BIN = QWEN_VENV / "bin"
QWEN_PY = QWEN_BIN / "python"
QWEN_EXE = QWEN_BIN / "vllm"
# Тот же порядок и флаги, что и для Higgs (vllm-omni#895 добавляет Qwen3-TTS в тот же
# пакет) — отдельный venv, чтобы падение установки одного бэкенда не блокировало другой.
QWEN_INSTALL_STEPS = [["vllm==0.26.0", "--torch-backend=auto"], ["vllm-omni==0.26.0"]]
QWEN_INSTALL_LOG = METRICS_DIR / "tts_qwen_install.log"

qwen_gate = None
if not RUN_QWEN:
    qwen_gate = "RUN_QWEN=False"
elif GPU is None:
    qwen_gate = "нет CUDA-устройства"

if qwen_gate:
    print(f"Qwen3-TTS стек не устанавливается: {qwen_gate}")
elif not INSTALL_QWEN_STACK and not QWEN_EXE.exists():
    qwen_gate = "окружение Qwen отсутствует, а INSTALL_QWEN_STACK=False"
    print(f"Qwen: {qwen_gate}")
else:
    try:
        ensure_venv(QWEN_VENV, QWEN_INSTALL_STEPS, installer="uv", log=QWEN_INSTALL_LOG)
    except Exception as error:
        qwen_gate = f"установка стека vLLM-Omni для Qwen не удалась: {error!r}"
        print(f"\n❌ {qwen_gate}")
    else:
        if QWEN_EXE.exists():
            print(f"\n✅ Стек vLLM-Omni установлен: {QWEN_EXE}")
        else:
            qwen_gate = f"установка прошла, но {QWEN_EXE.name} не появился в {QWEN_BIN}"
            print(f"\n❌ {qwen_gate}")


In [ ]:
# Один сервер на вариант чекпоинта: `task_type` работает только с той моделью,
# для которой он обучен (см. capability-матрицу в docs/research/qwen3-tts-notes.md).
qwen_metrics_by_variant = {}
qwen_status_by_variant = {}
qwen_port = 8001

for variant in QWEN_MODEL_VARIANTS:
    metrics_path = METRICS_DIR / f"tts_qwen_{variant}.json"
    if qwen_gate:
        status = f"SKIPPED ({qwen_gate})"
        skip_report = {"test": "tts_qwen_cuda", "model_variant": variant,
                       "status": "SKIPPED", "reason": qwen_gate, "results": []}
        if QWEN_INSTALL_LOG.exists():
            skip_report["install_log"] = str(QWEN_INSTALL_LOG)
            skip_report["install_log_tail"] = QWEN_INSTALL_LOG.read_text(
                errors="replace").splitlines()[-60:]
        metrics_path.write_text(json.dumps(skip_report, ensure_ascii=False, indent=2),
                                encoding="utf-8")
    else:
        command = [str(QWEN_PY), "src/tts_qwen_cuda.py",
                   "--model-variant", variant,
                   "--output-dir", str(OUTPUT_DIR),
                   "--metrics", str(metrics_path),
                   "--text-file", str(TTS_TEXT),
                   "--voice", QWEN_VOICE,
                   "--port", str(qwen_port),
                   "--max-new-tokens", str(QWEN_MAX_NEW_TOKENS)]
        if REF_WAV.exists() and REF_TXT.exists():
            command += ["--ref-audio", str(REF_WAV), "--ref-text", str(REF_TXT)]
        if QWEN_MEM_FRACTION_STATIC is not None:
            command += ["--mem-fraction-static", str(QWEN_MEM_FRACTION_STATIC)]
        if QWEN_MIN_CAPABILITY:
            command += ["--min-capability", QWEN_MIN_CAPABILITY]
        if QWEN_DEPLOY_CONFIG:
            command += ["--deploy-config", str(QWEN_DEPLOY_CONFIG)]
        for server_arg in QWEN_SERVER_ARGS:
            command += ["--server-arg", server_arg]
        for server_env in QWEN_SERVER_ENV:
            command += ["--server-env", server_env]

        # PATH нужен, чтобы раннер нашёл `vllm` из своего venv.
        environment = {**os.environ, "PATH": f"{QWEN_BIN}:{os.environ['PATH']}"}
        returncode = run(command, cwd=str(REPO), env=environment)
        status = "PASSED" if returncode == 0 else f"FAILED (exit {returncode})"
        qwen_port += 1

    qwen_status_by_variant[variant] = status
    qwen_metrics_by_variant[variant] = read_metrics(metrics_path) if metrics_path.exists() else None
    print(f"\nQwen3-TTS [{variant}]: {status}")

run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")


In [ ]:
print("Qwen3-TTS — независимый бэкенд: его статус не переносится на Higgs, и наоборот.")

for variant, metrics in qwen_metrics_by_variant.items():
    print(f"\n=== Qwen3-TTS [{variant}] ===")
    if not metrics:
        print(f"метрик нет: {qwen_status_by_variant[variant]}")
        continue
    if metrics.get("status") == "SKIPPED":
        print(f"SKIPPED: {metrics['reason']}")
        for line in metrics.get("install_log_tail", [])[-20:]:
            print("  " + line)
        continue
    if metrics.get("status") == "FAILED":
        print(f"FAILED: {metrics.get('exception')}")
        server_log = Path(metrics.get("server_log")
                          or (OUTPUT_DIR / f"qwen_vllm_server_{variant}.log"))
        if server_log.exists():
            print(f"\n--- последние строки {server_log.name} ---")
            print("\n".join(server_log.read_text(errors="replace").splitlines()[-40:]))

    for result in metrics.get("results", []):
        if result["status"] != "PASSED":
            print(f"{result['name']}: {result['status']} — "
                  f"{result.get('reason') or result.get('exception')}")
            continue
        print(f"{result['name']}: {result['audio_duration_seconds']:.2f} с аудио за "
              f"{result['processing_seconds']:.2f} с → RTF {result['rtf']:.3f}")
        display(Audio(filename=result["output"]))


## 4. STT: `bosonai/higgs-audio-v3-stt`

Вторичный этап. Окружение переиспользует предустановленный в Colab `torch`, доустанавливая
только несовместимый с системным `transformers==4.51.0`. Ревизия чекпоинта закреплена в
`src/stt_helper.py`.

In [ ]:
STT_VENV = Path("/content/venv-stt")
STT_PY = STT_VENV / "bin/python"
STT_PACKAGES = [["transformers==4.51.0", "tokenizers<0.22", "accelerate>=0.26.0",
                 "huggingface_hub<1.0", "soundfile", "librosa", "jiwer", "sentencepiece"]]

stt_gate = None
if not RUN_STT:
    stt_gate = "RUN_STT=False"
else:
    # Ошибки не поднимаются наружу: иначе Run all остановится и ВМ не отключится.
    try:
        ensure_venv(STT_VENV, STT_PACKAGES, log=METRICS_DIR / "stt_install.log")
    except Exception as error:
        stt_gate = f"установка окружения STT не удалась: {error!r}"
        note(f"❌ {stt_gate}")
    else:
        probe = ("import torch, transformers, huggingface_hub as hub; "
                 "print('torch', torch.__version__, '(', torch.__file__, ')'); "
                 "print('transformers', transformers.__version__, '| hub', hub.__version__, "
                 "'| cuda', torch.cuda.is_available())")
        if run([str(STT_PY), "-c", probe]) != 0:
            stt_gate = ("окружение STT не видит torch; пути родительского интерпретатора: "
                        f"{python_paths(sys.executable)}")
            note(f"❌ {stt_gate}")

In [ ]:
STT_METRICS = METRICS_DIR / "stt_cuda.json"
stt_status = "NOT RUN"

if stt_gate:
    stt_status = f"SKIPPED ({stt_gate})"
elif GPU is None:
    stt_status = "SKIPPED (нет CUDA-устройства)"
elif not STT_INPUT.exists():
    stt_status = f"SKIPPED (нет входа {STT_INPUT})"
else:
    # ffmpeg предустановлен в Colab; чекпоинт ожидает моно 16 кГц.
    normalized = Path("/content/stt_ru_16k.wav")
    normalized_ok = run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                         "-i", str(STT_INPUT), "-ac", "1", "-ar", "16000",
                         str(normalized)]) == 0

    command = [str(STT_PY), "src/stt_test.py",
               "--device", "cuda", "--dtype", STT_DTYPE,
               "--audio", str(normalized),
               "--output", str(OUTPUT_DIR / "stt_ru_colab.txt"),
               "--metrics", str(STT_METRICS)]
    if STT_REFERENCE.exists():
        command += ["--reference", str(STT_REFERENCE)]

    if not normalized_ok:
        stt_status = f"FAILED (ffmpeg не смог привести {STT_INPUT.name} к моно 16 кГц)"
    else:
        # Отдельный процесс: по его завершении вся VRAM и RAM модели возвращаются системе.
        returncode = run(command, cwd=str(REPO))
        stt_status = "PASSED" if returncode == 0 else f"FAILED (exit {returncode})"

print(f"\nSTT: {stt_status}")
if torch.cuda.is_available():
    # Ядро блокнота не аллоцировало модель, поэтому здесь ожидается около нуля.
    print(f"VRAM, удерживаемая ядром блокнота: "
          f"{torch.cuda.memory_allocated() / (1024 ** 3):.3f} GB")
run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

In [ ]:
stt_metrics = read_metrics(STT_METRICS) if STT_METRICS.exists() else None

if stt_metrics and stt_metrics.get("status") == "PASSED":
    print("📝 Транскрипция:\n" + stt_metrics["transcript"] + "\n")
    print(f"Устройство:      {stt_metrics['cuda_device']} ({stt_metrics['dtype']})")
    print(f"Ревизия модели:  {stt_metrics['revision']}")
    print(f"Загрузка:        {stt_metrics['model_load_seconds']:.2f} с")
    print(f"Длит. аудио:     {stt_metrics['audio_duration_seconds']:.2f} с")
    print(f"Обработка:       {stt_metrics['processing_seconds']:.2f} с")
    print(f"RTF:             {stt_metrics['rtf']:.3f}")
    if stt_metrics.get("wer") is None:
        print(f"WER:             не измерен ({stt_metrics.get('wer_note')})")
    else:
        print(f"WER:             {stt_metrics['wer']:.4f} "
              f"(референс: {stt_metrics['wer_reference']})")
    print(f"Пик VRAM:        {stt_metrics['peak_vram_bytes'] / (1024 ** 3):.2f} GB")
    print(f"Пик RSS:         {stt_metrics['peak_host_rss_bytes'] / (1024 ** 3):.2f} GB")
elif stt_metrics:
    print(f"STT {stt_metrics.get('status')}: {stt_metrics.get('exception')}")
    print(stt_metrics.get("traceback", ""))
else:
    print(f"STT метрик нет: {stt_status}")

## 5. Сводка

Таблица собирается только из записанных метрик: там, где этап не выполнялся, стоит его
фактический статус, а не подставленное значение.

In [ ]:
import pandas as pd

# Опорные значения локального замера на Apple Silicon M1 (16 ГБ, macOS 14.6.1).
M1_RTF = {"tts_basic": 7.02, "tts_controls": 12.61, "tts_clone": 822.09, "stt": 1.40}

rows = []
tts_results = {r["name"]: r for r in (tts_metrics or {}).get("results", [])}
device_peak = (tts_metrics or {}).get("peak_device_vram_bytes")
for name in ("tts_basic", "tts_controls", "tts_clone"):
    result = tts_results.get(name)
    if result and result["status"] == "PASSED":
        rows.append({"Этап": name, "Статус": "PASSED",
                     "RTF (Colab)": f"{result['rtf']:.3f}",
                     "RTF (M1)": f"{M1_RTF[name]:.2f}",
                     # Веса TTS живут в процессе сервера, поэтому пик замеряется
                     # device-wide через nvidia-smi, а не через torch раннера.
                     "Пик VRAM, GB": f"{device_peak / (1024 ** 3):.2f}*" if device_peak else "—",
                     "Артефакт": result["output"]})
    else:
        status = (result or {}).get("status") or (tts_metrics or {}).get("status") or tts_status
        reason = (result or {}).get("reason") or (tts_metrics or {}).get("reason") or ""
        rows.append({"Этап": name, "Статус": f"{status}{': ' + reason if reason else ''}",
                     "RTF (Colab)": "—", "RTF (M1)": f"{M1_RTF[name]:.2f}",
                     "Пик VRAM, GB": "—", "Артефакт": "—"})

# Qwen3-TTS — отдельные строки на каждый (вариант, тест); статус читается
# независимо от Higgs, провал одного бэкенда не переносится на другой.
for variant, q_metrics in (qwen_metrics_by_variant or {}).items():
    variant_results = {r["name"]: r for r in (q_metrics or {}).get("results", [])}
    variant_peak = (q_metrics or {}).get("peak_device_vram_bytes")
    if variant_results:
        for name, result in variant_results.items():
            label = f"Qwen[{variant}]:{name}"
            if result["status"] == "PASSED":
                rows.append({"Этап": label, "Статус": "PASSED",
                             "RTF (Colab)": f"{result['rtf']:.3f}", "RTF (M1)": "—",
                             "Пик VRAM, GB": f"{variant_peak / (1024 ** 3):.2f}*" if variant_peak else "—",
                             "Артефакт": result["output"]})
            else:
                reason = result.get("reason") or result.get("exception") or ""
                rows.append({"Этап": label,
                             "Статус": f"{result['status']}{': ' + reason if reason else ''}",
                             "RTF (Colab)": "—", "RTF (M1)": "—",
                             "Пик VRAM, GB": "—", "Артефакт": "—"})
    else:
        status = (q_metrics or {}).get("status") or qwen_status_by_variant.get(variant, "NOT RUN")
        reason = (q_metrics or {}).get("reason") or ""
        rows.append({"Этап": f"Qwen[{variant}]",
                     "Статус": f"{status}{': ' + reason if reason else ''}",
                     "RTF (Colab)": "—", "RTF (M1)": "—",
                     "Пик VRAM, GB": "—", "Артефакт": "—"})

if stt_metrics and stt_metrics.get("status") == "PASSED":
    rows.append({"Этап": "STT", "Статус": "PASSED",
                 "RTF (Colab)": f"{stt_metrics['rtf']:.3f}",
                 "RTF (M1)": f"{M1_RTF['stt']:.2f}",
                 "Пик VRAM, GB": f"{stt_metrics['peak_vram_bytes'] / (1024 ** 3):.2f}",
                 "Артефакт": stt_metrics["output"]})
else:
    rows.append({"Этап": "STT", "Статус": stt_status, "RTF (Colab)": "—",
                 "RTF (M1)": f"{M1_RTF['stt']:.2f}", "Пик VRAM, GB": "—", "Артефакт": "—"})

display(pd.DataFrame(rows))
print("TTS (*): пик по устройству целиком (nvidia-smi), пока жил сервер. "
      "STT: аллокации процесса STT (torch).")

report = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S%z"),
    "platform": "Google Colab",
    "repo_revision": REPO_SHA,
    "gpu": GPU,
    "notebook_python": sys.version.split()[0],
    "notebook_torch": torch.__version__,
    "tts_backend": TTS_BACKEND,
    "tts": tts_metrics or {"status": tts_status},
    "stt": stt_metrics or {"status": stt_status},
    "qwen_tts": qwen_metrics_by_variant,
}
report_path = METRICS_DIR / "benchmark_colab_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str),
                       encoding="utf-8")
print(f"\n✅ Полный отчёт: {report_path}")

In [ ]:
# ── Завершение: сброс на Диск, затем безусловное отключение ──
# Модели жили в дочерних процессах, которые уже завершились, поэтому выгружать
# из ядра нечего. Остаётся довести данные до Диска и освободить квоту.
try:
    print(f"На Диске сохранено: {WORKSPACE}")
    for artefact in sorted(METRICS_DIR.iterdir()) + sorted(OUTPUT_DIR.iterdir()):
        if artefact.is_file() and not artefact.name.startswith("."):
            print(f"  - {artefact.relative_to(WORKSPACE)} "
                  f"({artefact.stat().st_size / 1024:.1f} KB)")
except Exception as error:
    print(f"не удалось перечислить артефакты: {error!r}")

try:
    if USE_DRIVE:
        from google.colab import drive
        drive.flush_and_unmount()
        print("💾 Данные синхронизированы: MyDrive/higgs-benchmark/")
finally:
    # Безусловно: квота Colab не должна гореть на простое. Разбор результатов —
    # по файлам на Диске, а не по выводу ячеек.
    from google.colab import runtime
    print("🛑 Отключение ВМ...")
    runtime.unassign()